# Pipeline of execution: 

* Run `resample.py`
* Run `features_volumetric.py`
* Run `features_radiomic.py`
* Run `features_displacement.py`

In [1]:
# geração dos extremos sMCI/pMCI

# import pandas as pd

# INPUT_PATH = "/mnt/study-data/pgirardi/graphs/image_data_sMCI_pMCI.txt"
# OUTPUT_PATH = "/mnt/study-data/pgirardi/graphs/image_data_sMCI_pMCI_extremos.csv"

# df = pd.read_csv(INPUT_PATH)
# df["MRI_DATE"] = pd.to_datetime(df["MRI_DATE"])

# parts = []
# for id_pt, g in df.groupby("ID_PT", sort=False):
#     g_smci = g.loc[g["GROUP"] == "sMCI"].sort_values("MRI_DATE", ascending=True)
#     early_smci = g_smci.head(3).copy()

#     g_pmci = g.loc[g["GROUP"] == "pMCI"].sort_values("MRI_DATE", ascending=True)
#     late_pmci = g_pmci.tail(3).copy()

#     parts.extend([early_smci, late_pmci])

# df_ext = pd.concat(parts, ignore_index=True)
# df_ext.to_csv(OUTPUT_PATH, index=False)

# print("Salvo:" , OUTPUT_PATH)
# print("df_ext:", df_ext.shape, "linhas |", df_ext["ID_PT"].nunique(), "pacientes")
# df_ext.head(10)

In [2]:
import pandas as pd

# df_ext já carregado; se não:
df_ext = pd.read_csv("image_data_sMCI_pMCI_extremos.csv")

# 1 linha por paciente (metadados do conjunto)
pt = (
    df_ext.sort_values(["ID_PT", "MRI_DATE", "ID_IMG"])
    .groupby("ID_PT", as_index=False)
    .agg(
        GROUP=("GROUP", "first"),
        SEX=("SEX", "first"),
        n_linhas=("ID_IMG", "size"),
    )
)

# conjuntos = pacientes com exatamente 3 linhas
pt = pt[pt["n_linhas"] == 3].copy()
pt["n_conjuntos"] = 1  # 1 conjunto por paciente válido

# --- totais ---
n_conjuntos_total = len(pt)
n_linhas_total = int(pt["n_linhas"].sum())
print("Conjuntos (pacientes com 3 linhas):", n_conjuntos_total)
print("Linhas totais:", n_linhas_total)
print("Checagem linhas/3:", n_linhas_total / 3)

# --- por GROUP ---
print("\nPor GROUP:")
print(pt["GROUP"].value_counts().sort_index())
# ou só contagem de conjuntos:
# print(pt.groupby("GROUP")["n_conjuntos"].sum())

# --- por SEX ---
print("\nPor SEX:")
print(pt["SEX"].value_counts().sort_index())

# --- GROUP × SEX ---
print("\nGROUP × SEX:")
print(pd.crosstab(pt["GROUP"], pt["SEX"], margins=True))

Conjuntos (pacientes com 3 linhas): 525
Linhas totais: 1575
Checagem linhas/3: 525.0

Por GROUP:
GROUP
pMCI    128
sMCI    397
Name: count, dtype: int64

Por SEX:
SEX
F    222
M    303
Name: count, dtype: int64

GROUP × SEX:
SEX      F    M  All
GROUP               
pMCI    54   74  128
sMCI   168  229  397
All    222  303  525


In [3]:
# Leitura radiomicos e volumetricos

import pandas as pd

ab = "abordagem_4_sMCI_pMCI_extremos"

vol_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/features_volumetric.csv"

rad_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/features_radiomic.csv"

vol_df = pd.read_csv(vol_path)
rad_df = pd.read_csv(rad_path)


In [4]:
# Normalizacao pelo ICV (mask_mm3 global) e une radiomicos com volumericos

# ICV = volume total da mascara do encefalo (linha __global__ em features_volumetric)
# Vamos anexar o ICV por ID_IMG e normalizar apenas as features dependentes de tamanho.

import numpy as np

# 1) ICV por ID_IMG
vol_df["ID_IMG"] = vol_df["ID_IMG"].astype(str).str.strip()

icv_df = (
    vol_df.loc[vol_df["roi"].astype(str) == "__global__", ["ID_IMG", "mask_mm3"]]
    .drop_duplicates(subset=["ID_IMG"], keep="last")
    .rename(columns={"mask_mm3": "ICV_mask_mm3"})
)

icv_df["ICV_mask_mm3"] = pd.to_numeric(icv_df["ICV_mask_mm3"], errors="coerce")

# 2) Merge com radiomicos
rad_df["ID_IMG"] = rad_df["ID_IMG"].astype(str).str.strip()
merged = rad_df.merge(icv_df, on="ID_IMG", how="left", validate="many_to_one")

missing_icv = int(merged["ICV_mask_mm3"].isna().sum())
if missing_icv:
    raise ValueError(
        f"ICV ausente para {missing_icv} linhas radiomicas. "
        "Verifique se todos os ID_IMG do radiomico existem no volumetrico (linha __global__)."
    )

# 3) Features a normalizar (dependentes de tamanho)
cols_to_norm = [
    # first-order
    "original_firstorder_Energy",
    "original_firstorder_TotalEnergy",
    # shape (absolutas)
    "original_shape_MeshVolume",
    "original_shape_VoxelVolume",
    "original_shape_SurfaceArea",
    "original_shape_LeastAxisLength",
    "original_shape_MajorAxisLength",
    "original_shape_MinorAxisLength",
    "original_shape_Maximum2DDiameterColumn",
    "original_shape_Maximum2DDiameterRow",
    "original_shape_Maximum2DDiameterSlice",
    "original_shape_Maximum3DDiameter",
]

# mantém só as que existem no arquivo (robustez)
cols_to_norm = [c for c in cols_to_norm if c in merged.columns]

# garante numérico
for c in cols_to_norm:
    merged[c] = pd.to_numeric(merged[c], errors="coerce")

# 4) Normaliza pelo ICV
# Observacao: algumas features (p.ex. area) ficam com unidade relativa a volume;
# aqui seguimos a definicao pedida: dividir pelo volume total do encefalo.
merged[cols_to_norm] = merged[cols_to_norm].div(merged["ICV_mask_mm3"], axis=0)

# 5) CSV final: sem colunas "brutas" não-normalizadas adicionais (mantemos só a versão normalizada)
# (as colunas normalizadas sobrescrevem as originais, então basta salvar o merged)
out_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/radiomics_merge.csv"
merged.to_csv(out_path, index=False)

out_path


'/mnt/study-data/pgirardi/graphs/csvs/abordagem_4_sMCI_pMCI_extremos/radiomics_merge.csv'

In [19]:
# Leitura dados técnicos e inserção dos equipamentos na planilha de atributos radiomicos

import pandas as pd

tech_path = f"/mnt/study-data/pgirardi/graphs/csvs/adnimerged.csv"

radiomics_merged_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/radiomics_merge.csv"

tech_df = pd.read_csv(tech_path)
radiomics_merged_df = pd.read_csv(radiomics_merged_path)

In [21]:
# Inserção de variáveis técnicas (por ID_IMG)

import numpy as np
import pandas as pd

# A planilha "merge" será o radiomics_merged_df enriquecido com informações técnicas/clínicas.
# (Se você preferir outro nome, basta trocar a variável aqui.)
merge = radiomics_merged_df.copy()

# 1) Merge por ID_IMG trazendo as colunas necessárias
cols_needed = [
    "ID_IMG",
    "ID_PT",
    "SEX",
    "DIAG",
    "MRI_DATE",
    "FIELD_STRENGTH",
    #"SLICE_THICKNESS",
    "MANUFACTURER",
    "MFG_MODEL",
    "MMSE_SCORE",
    "CDR_GLOBAL",
    "ADAS_SCORE",
    "FAQ_SCORE",
]

tech_sub = tech_df.loc[:, cols_needed].copy()
tech_sub["ID_IMG"] = tech_sub["ID_IMG"].astype(str).str.strip()
tech_sub = tech_sub.drop_duplicates(subset=["ID_IMG"], keep="last")

merge["ID_IMG"] = merge["ID_IMG"].astype(str).str.strip()
merge = merge.merge(tech_sub, on="ID_IMG", how="left", validate="many_to_one")

# 2) Define batch (scanner) a partir das variáveis do equipamento
merge["batch"] = (
    merge[["MANUFACTURER", "MFG_MODEL", "FIELD_STRENGTH"]]#, "SLICE_THICKNESS"]]
    .astype(str)
    .agg("_".join, axis=1)
)

missing_any = int(merge[cols_needed[1:]].isna().any(axis=1).sum())
print(f"Linhas sem alguma info técnica/clínica necessária (após merge): {missing_any}")



Linhas sem alguma info técnica/clínica necessária (após merge): 0


In [22]:
import pandas as pd

disp_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/features_displacement.csv"
out_all_unit = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/all_features_merge.csv"

MERGE_KEYS = ["ID_IMG", "roi", "side", "label"]
# Metadados clínicos vêm do displacement; rad fica com radiomico + ICV + técnicas + batch
OVERLAP_META = {"ID_PT", "SEX", "DIAG", "GROUP", "AGE", "MRI_DATE"}


def normalize_merge_keys(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["ID_IMG"] = out["ID_IMG"].astype(str).str.strip()
    out["roi"] = out["roi"].astype(str).str.strip()
    out["side"] = out["side"].astype(str).str.strip()
    out["label"] = out["label"].astype(str).str.strip()
    return out


# merge = radiomico + ICV + FIELD_STRENGTH, MANUFACTURER, ... + batch (célula anterior)
rad = merge.drop(columns=[c for c in OVERLAP_META if c in merge.columns], errors="ignore")
rad = normalize_merge_keys(rad)

disp = pd.read_csv(disp_path)
disp = normalize_merge_keys(disp)

print("rad (com técnicas):", rad.shape)
print("disp:", disp.shape)

all_unit = rad.merge(
    disp,
    on=MERGE_KEYS,
    how="left",
    validate="one_to_one",
)

if "MRI_DATE" in all_unit.columns:
    all_unit["MRI_DATE"] = (
        pd.to_datetime(all_unit["MRI_DATE"], errors="coerce").dt.strftime("%Y-%m-%d")
    )


def reorder_all_features_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Demográficas/técnicas primeiro; depois radiomico, ICV e deslocamento."""
    meta_cols = [
        "ID_IMG",
        "ID_PT",
        "GROUP",
        "SEX",
        "AGE",
        "MRI_DATE",
        "roi",
        "side",
        "label",
        "FIELD_STRENGTH",
        #"SLICE_THICKNESS",
        "MANUFACTURER",
        "MFG_MODEL",
        "batch",
        "DIAG",
        #"ref_tag",
        "MMSE_SCORE",
        "CDR_GLOBAL",
        "ADAS_SCORE",
        "FAQ_SCORE",
    ]
    disp_prefixes = (
        "centroid_",
        "logjac_",
        "mag_",
        "div_",
        "ux_",
        "uy_",
        "uz_",
        "curlmag_",
        "strain_trace_",
        "strain_det_",
        "strain_fro_",
        "strain_vol_",
        "strain_dev_norm_",
        "strain_shear_max_",
        "strain_l1_",
        "strain_l2_",
        "strain_l3_",
        "strain_shear_ratio_",
        "strain_shear_energy_",
    )
    stat_suffixes = ("_n", "_mean", "_std", "_p05", "_p50", "_p95")

    cols = list(df.columns)
    prefix = [c for c in meta_cols if c in cols]
    radiomic = [c for c in cols if c.startswith("original_")]
    icv = [c for c in cols if c == "ICV_mask_mm3"]
    used = set(prefix) | set(radiomic) | set(icv)
    disp_cols = [c for c in cols if c not in used]

    def _disp_key(name: str) -> tuple:
        for i, p in enumerate(disp_prefixes):
            if name.startswith(p):
                base = p.rstrip("_")
                for j, suf in enumerate(stat_suffixes):
                    if name.endswith(suf) and name == base + suf:
                        return (i, j, name)
                return (i, 99, name)
        return (len(disp_prefixes), 0, name)

    disp_cols = sorted(disp_cols, key=_disp_key)
    ordered = prefix + radiomic + icv + disp_cols
    extra = [c for c in cols if c not in ordered]
    if extra:
        print(f"Aviso: colunas extras ao final: {extra}")
        ordered = ordered + extra
    return df.loc[:, ordered]


all_unit = reorder_all_features_columns(all_unit)

all_unit.to_csv(out_all_unit, index=False)
print("Salvo:", out_all_unit, "| shape:", all_unit.shape)

rad (com técnicas): (31500, 120)
disp: (31500, 122)
Salvo: /mnt/study-data/pgirardi/graphs/csvs/abordagem_4_sMCI_pMCI_extremos/all_features_merge.csv | shape: (31500, 238)


# Pré-processamento da planilha

In [23]:
all_features_merge = pd.read_csv("csvs/abordagem_4_sMCI_pMCI_extremos/all_features_merge.csv")

# remove colunas que não fazem mais sentido
all_features_merge = all_features_merge.drop(columns=["ID_IMG", "MRI_DATE", "AGE", "label","roi", "side", "FIELD_STRENGTH", "MANUFACTURER", "MFG_MODEL", "batch", "DIAG", "MMSE_SCORE", "CDR_GLOBAL", "ADAS_SCORE", "FAQ_SCORE", "ref_tag"])

# converte para 0/1
all_features_merge["SEX"] = all_features_merge["SEX"].map({"M": 0, "F": 1})
all_features_merge["GROUP"] = all_features_merge["GROUP"].map({"sMCI": 0, "pMCI": 1})

all_features_merge.to_csv("csvs/abordagem_4_sMCI_pMCI_extremos/all_features_merge_clean.csv", index=False)

